In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
importlib.reload(gpt)

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6'

In [ ]:
main_df = pd.read_excel(os.path.join(DIR, '1556_with_eligibility_imp.xlsx'))
main_df.head()

In [ ]:
main_df.columns

In [ ]:
small_df = pd.read_excel(os.path.join(DIR, '86_for review.xlsx'))
small_df.columns

In [ ]:
small_df['title_extract'] = small_df.Title.apply(extract_letters)

In [ ]:
main_df['in_86'] = main_df.title_extract.isin(small_df.title_extract)
main_df['in_86'].sum()

In [ ]:
small_df['in_1500'] = small_df.title_extract.isin(main_df.title_extract)

In [ ]:
small_df.to_excel(os.path.join(DIR, '86_with_inclusion.xlsx'), index=False)

## Run model on titles only

In [ ]:
main_copy = main_df.copy()
del main_copy['abstract']
main_copy['abstract'] = np.nan
eligibility_df = gpt.check_eligibility(main_copy, criteria=gpt.JUMALAH_CRITERIA, batch_size=100)

In [ ]:
eligibility_df.decision.sum(), eligibility_df.decision.mean(), eligibility_df.decision.isna().sum()

In [ ]:
part = eligibility_df[['reasoning', 'decision']]
part.columns = ['reasoning_title', 'decision_title']
main_df_updated = pd.concat([main_df, part], axis=1)
main_df_updated.loc[1]

In [ ]:
main_df_updated.decision_title.sum(), main_df_updated.decision_title_abstract.sum()

In [ ]:
(main_df_updated.decision_title != main_df_updated.decision_title_abstract).sum(), len(main_df_updated)

In [ ]:
main_df_updated.to_excel(os.path.join(DIR, '1556_with_eligibility_title_abstract.xlsx'), index=False)